# Lab 3 — Fintech Research & Risk Analysis Agent

This notebook builds a LangGraph multi-agent workflow that answers fintech research questions using local PDF RAG (Chroma) and a local SQLite database, with Groq-powered Writer and QA agents. It's designed to plug into MCP servers (PDF search, financial metrics, risk factors) for production use.

## Concepts covered

- PDF RAG: `PyPDFLoader` -> chunking -> local embeddings -> Chroma
- Structured data: local SQLite (`companies`, `financial_metrics`, `products`, `risks`)
- LangGraph supervisor with fan-out/fan-in (`Send`, state reducers)
- Groq-powered Writer and QA agents
- MCP tool shape for production: `search_documents`, `get_financial_metrics`, `get_risk_factors`
- Docker deployment + bearer-token auth notes for remote MCP servers

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if (ROOT / 'src').exists():
    sys.path.insert(0, str(ROOT))
else:
    sys.path.insert(0, str(ROOT.parent))

from dotenv import load_dotenv
load_dotenv()

from src.agents.langgraph_agent import (
    build_supervisor_graph,
    run_supervisor_demo,
    init_fintech_db,
    ingest_pdfs,
)
print('Fintech lab imports ready')

C:\Users\MadhiarasanM\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fintech lab imports ready


## Set up local data

1. Create the local SQLite schema (seeded with a `DemoFintech Ltd` example if empty).
2. Drop your own fintech PDFs (annual reports, investor decks, risk disclosures) into `data/pdfs/`, then index them into a local Chroma store.

Everything below runs locally — Groq is only used for the LLM calls; embeddings and the vector store stay on disk.

In [2]:
init_fintech_db()
print('SQLite schema ready at data/fintech.db (seeded with DemoFintech Ltd)')

SQLite schema ready at data/fintech.db (seeded with DemoFintech Ltd)


## Index your PDFs

Run this once you've added PDFs to `data/pdfs/`. The PDF researcher node also calls this lazily on first use if the Chroma store doesn't exist yet.

In [4]:
try:
    ingest_pdfs()
    print('Indexed PDFs from data/pdfs into data/chroma_fintech')
except FileNotFoundError as e:  
    print(e)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3148.39it/s]


Indexed PDFs from data/pdfs into data/chroma_fintech


## Connect multiple MCP servers

In the real exercise, you wire `pdf_search_server.py` (`search_documents`), `financial_metrics_server.py` (`get_financial_metrics`), and `risk_server.py` (`get_risk_factors`) through `MultiServerMCPClient`, then pass the returned tools into a LangGraph or LangChain agent.

### Current learning path

1. Start the financial-metrics MCP server (wraps the SQLite tables).
2. Start the PDF-search MCP server (wraps the Chroma retriever).
3. Start the risk-factors MCP server.
4. Build the multi-server fintech agent.
5. Run the supervisor graph below.

In [5]:
result = run_supervisor_demo(
    "What are the company's biggest risks?",
    company="DemoFintech Ltd",
)
print(result['final'])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3730.62it/s]


# Executive Summary

**Executive Summary for DemoFintech Ltd**

DemoFintech Ltd operates in a complex and rapidly evolving fintech landscape, exposing the company to various risks that could impact its stability and success. The company's biggest risks can be broadly categorized into regulatory, operational, and financial risks. Regulatory changes, particularly those related to the UPI fee structure, pose a high-severity risk to the company. This could significantly impact DemoFintech's revenue streams and profitability.

In addition to regulatory risks, cybersecurity and data-protection exposure are also significant concerns for the company. Although the severity of this risk is medium, the potential consequences of a significant cybersecurity incident could be severe, compromising sensitive customer data and damaging the company's reputation. The company must prioritize robust cybersecurity measures to mitigate this risk and protect its assets.

The company is also exposed to various

## Production notes

- `MultiServerMCPClient` is the adapter that combines the PDF-search, financial-metrics, and risk-factors MCP servers.
- Auth bearer tokens belong in the HTTP transport layer for remote servers.
- Docker makes it easy to deploy each server (and the Chroma/SQLite volumes) independently.

## Edit points

Modify `pdf_researcher` / `sqlite_researcher` in `src/agents/langgraph_agent.py` to change retrieval logic or add new tables, and wrap `query_financial_metrics`, `query_risks`, and the Chroma retriever as `@mcp.tool()` functions in `src/servers/` to expose them over MCP.